# AI Model Experiment & Evaluation
**Sentiment Analysis Ulasan Pelanggan — Scikit-learn vs Gemini API**

Notebook ini membandingkan dua pendekatan AI untuk klasifikasi sentimen ulasan e-commerce:
1. Model klasik Machine Learning (TF-IDF + Logistic Regression, Scikit-learn)
2. LLM API (Gemini, prompting zero-shot)

Peserta: Adighani Ilham (CAI58) · ReWork Academy AI Engineering Bootcamp Batch 1

## 1. Problem Statement

- **Objective:** memilih pendekatan AI yang paling layak untuk mengklasifikasi sentimen ulasan pelanggan toko online secara otomatis.
- **Target/label:** `sentiment` ∈ {`positif`, `negatif`} (klasifikasi biner). Input: `review_text` (bahasa Indonesia, kalimat pendek).
- **Asumsi:** label di dataset dianggap *ground truth*; kelas positif/negatif dianggap sama penting (jadi metrik utama = Accuracy + macro-F1); tidak ada kelas netral.
- **Batasan:** dataset kecil (200 baris), satu domain (e-commerce), satu bahasa. Hasil adalah **eksperimen kecil**, bukan klaim performa produksi.
- **Temuan data penting (dibuktikan di Section 3):** dari 200 baris, hanya **40 teks ulasan unik** — satu teks dipakai ulang ~5 kali dengan `product_name` berbeda. Ini membuat split acak biasa **bocor** (teks yang sama muncul di train dan test), sehingga evaluasi dilakukan dengan dua cara: split sesuai starter (untuk transparansi) dan split bebas-kebocoran (sebagai hasil utama).

## 2. Import Library & Load Dataset

In [ ]:
import os, re, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, LeaveOneOut
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (classification_report, confusion_matrix, accuracy_score,
                             precision_recall_fscore_support, ConfusionMatrixDisplay)
from google import genai
from google.genai import types
warnings.filterwarnings('ignore')

# Jalankan dari folder notebook/ (path relatif ke ../data)
DATA_PATH = '../data/customer_reviews_sentiment.csv'
df = pd.read_csv(DATA_PATH)
print(df.shape); print(df.sentiment.value_counts().to_dict())
df.head()

In [ ]:
# Pengecekan kualitas data
print('Missing values:', df.isna().sum().to_dict())
print('Teks unik   :', df.review_text.nunique(), 'dari', len(df), 'baris')
print('Baris duplikat teks:', df.review_text.duplicated().sum())
print('Teks dengan >1 label berbeda:', (df.groupby('review_text').sentiment.nunique() > 1).sum())
print(df.groupby('review_text').size().describe()[['min','mean','max']].to_dict())

## 3. Menyiapkan Train/Test Split

Test set yang **sama** dipakai untuk kedua pendekatan agar adil. Ada dua skema:

- **Split A (sesuai starter):** `train_test_split` stratified 80/20, `random_state=42`. Rawan bocor karena teks duplikat.
- **Split B (bebas kebocoran — hasil utama):** split per *teks unik*; seluruh baris dari satu teks masuk ke train **atau** test, tidak keduanya. Dipilih 4 teks positif + 4 teks negatif (20% dari 40 teks unik) sebagai test, `random_state=42`.

In [ ]:
X, y = df['review_text'], df['sentiment']

# --- Split A: sesuai starter ---
XA_tr, XA_te, yA_tr, yA_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
overlap_A = XA_te.isin(set(XA_tr)).mean()
print(f'Split A -> train {len(XA_tr)}, test {len(XA_te)}; % baris test yang teksnya juga ada di train: {overlap_A:.0%}')

# --- Split B: group split per teks unik ---
rng = np.random.RandomState(42)
uniq = df.drop_duplicates('review_text')[['review_text','sentiment']]
test_texts = []
for lab, grp in uniq.groupby('sentiment'):
    test_texts += list(rng.choice(grp.review_text.values, size=int(round(len(grp)*0.2)), replace=False))
mask_te = df.review_text.isin(test_texts)
XB_tr, yB_tr = df.loc[~mask_te,'review_text'], df.loc[~mask_te,'sentiment']
XB_te, yB_te = df.loc[mask_te,'review_text'],  df.loc[mask_te,'sentiment']
overlap_B = XB_te.isin(set(XB_tr)).mean()
print(f'Split B -> train {len(XB_tr)}, test {len(XB_te)} ({XB_te.nunique()} teks unik); overlap: {overlap_B:.0%}')
print('Distribusi test B:', yB_te.value_counts().to_dict())

## 4. Pendekatan 1 — Model Klasik (Scikit-learn)

### 4.1 Preprocessing & Feature Extraction
TF-IDF (unigram, default) — tanpa stemming/stopword agar sederhana & reproducible; teks sudah pendek dan bersih.

### 4.2 Training Model
Logistic Regression (default, `random_state=42`) — baseline linear yang cepat dan mudah diinterpretasi.

In [ ]:
def make_clf():
    return make_pipeline(TfidfVectorizer(), LogisticRegression(random_state=42, max_iter=1000))

clf_A = make_clf().fit(XA_tr, yA_tr)
clf_B = make_clf().fit(XB_tr, yB_tr)
print('Model klasik terlatih pada Split A dan Split B.')

### 4.3 Prediksi pada Test Set

In [ ]:
pred_classic_A = clf_A.predict(XA_te)
pred_classic_B = clf_B.predict(XB_te)

## 5. Pendekatan 2 — LLM API (Gemini)

### 5.1 Setup API
API key dibaca dari environment variable `GEMINI_API_KEY` (tidak ada key di dalam repo).

In [ ]:
client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])
GEMINI_MODEL = 'gemini-3.7-flash'   # lihat catatan di bawah
GEN_PARAMS = dict(temperature=0.0, max_output_tokens=2048)
print('Model:', GEMINI_MODEL, '| params:', GEN_PARAMS)

### 5.2 Merancang Prompt

**Pilihan prompt: zero-shot, dengan batch kecil (10 ulasan per request).**
- *Zero-shot* dipilih karena tugas sentimen sederhana & umum; LLM **tidak** melihat data latih sama sekali (kontras jelas dengan model klasik). Instruksi eksplisit: hanya dua label, output berupa JSON array berurutan.
- **temperature = 0.0** — klasifikasi butuh output deterministik/konsisten (starter memakai 0.2; diturunkan agar reproducible). `max_output_tokens=2048` dibuat longgar karena model flash terbaru memakai *thinking tokens*; batas kecil (64) pada percobaan awal membuat jawaban terpotong/kosong (bug pipeline, bukan kesalahan model → diperbaiki dan eksperimen diulang).
- **Batching (10 ulasan/request):** API free tier pada key ini dibatasi **20 request/hari/model**; memanggil 40 teks satu-per-satu (starter) tidak muat. Maka 40 teks unik dikirim dalam 4 request. Konsekuensi: sedikit perbedaan dari prompt per-ulasan, dicatat sebagai keterbatasan.
- **Model:** brief/starter menyebut `gemini-2.0-flash`, tetapi model itu sudah dipensiunkan di API (HTTP 404), begitu juga `gemini-2.5-flash` untuk key ini. Dipakai **`gemini-3.7-flash`**, model keluarga *flash* yang tersedia dan kuotanya masih ada.
- Output dinormalisasi (lowercase, strip); jawaban di luar `positif`/`negatif` atau jumlah item tidak cocok dihitung `invalid` dan **dicatat sebagai salah**, bukan dibuang.

In [ ]:
PROMPT = '''Klasifikasikan sentimen setiap ulasan pelanggan berikut sebagai "positif" atau "negatif".
Jawab HANYA dengan JSON array berisi string label ("positif" atau "negatif"), satu label per ulasan, sesuai urutan.

Ulasan:
{items}'''

def classify_batch(texts, retries=5):
    items = '\n'.join(f'{i+1}. "{t}"' for i, t in enumerate(texts))
    for i in range(retries):
        try:
            r = client.models.generate_content(
                model=GEMINI_MODEL, contents=PROMPT.format(items=items),
                config=types.GenerateContentConfig(response_mime_type='application/json', **GEN_PARAMS))
            return (r.text or '').strip()
        except Exception as e:
            if i == retries - 1: raise
            time.sleep(10 * 2 ** i)

def parse_batch(raw, n):
    try:
        arr = json.loads(raw)
        if isinstance(arr, list) and len(arr) == n:
            return [str(a).strip().lower() if str(a).strip().lower() in ('positif','negatif') else 'invalid' for a in arr]
    except Exception:
        pass
    return ['invalid'] * n

### 5.3 Menjalankan Prediksi pada Test Set
LLM dipanggil pada **40 teks unik** (4 request × 10 teks, output deterministik) dan hasilnya di-cache ke `../data/gemini_predictions.csv` agar hemat kuota & reproducible tanpa API key. Prediksi test set tiap split diambil dari tabel ini lewat pencocokan teks.

In [ ]:
CACHE = '../data/gemini_predictions.csv'
BATCH = 10
if os.path.exists(CACHE):
    cache = pd.read_csv(CACHE)
else:
    rows = []; texts = list(uniq.review_text)
    for i in range(0, len(texts), BATCH):
        chunk = texts[i:i+BATCH]; s = time.time()
        raw = classify_batch(chunk); dt = time.time() - s
        labs = parse_batch(raw, len(chunk))
        rows += [{'review_text': t, 'llm_label': l, 'batch_no': i//BATCH + 1, 'batch_latency_s': round(dt, 2)} for t, l in zip(chunk, labs)]
        time.sleep(3)
    cache = pd.DataFrame(rows); cache.to_csv(CACHE, index=False)
llm_map = dict(zip(cache.review_text, cache.llm_label))
print('Output invalid:', (cache.llm_label == 'invalid').sum(), 'dari', len(cache))
print('Request API:', cache.batch_no.nunique(), '| latensi rata-rata per request: %.1fs' % cache.drop_duplicates('batch_no').batch_latency_s.mean())

pred_llm_A = XA_te.map(llm_map).values
pred_llm_B = XB_te.map(llm_map).values

## 6. Evaluasi dan Perbandingan

### 6.1 Evaluasi Model Klasik

In [ ]:
LABELS = ['negatif', 'positif']
def evaluate(name, y_true, y_pred):
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, labels=LABELS, average='macro', zero_division=0)
    return {'Pendekatan': name, 'Accuracy': accuracy_score(y_true, y_pred), 'Precision (macro)': p, 'Recall (macro)': r, 'F1 (macro)': f}

for tag, yt, yp in [('Split A (starter, rawan bocor)', yA_te, pred_classic_A), ('Split B (bebas kebocoran) - UTAMA', yB_te, pred_classic_B)]:
    print(f'=== Model Klasik (Scikit-learn) — {tag} ===')
    print(classification_report(yt, yp, labels=LABELS, zero_division=0))
    print('Confusion Matrix [baris=aktual, kolom=prediksi; urutan negatif, positif]:\n', confusion_matrix(yt, yp, labels=LABELS))

### 6.2 Evaluasi LLM API

In [ ]:
LABELS_ALL = ['negatif', 'positif', 'invalid']
for tag, yt, yp in [('Split A', yA_te, pred_llm_A), ('Split B - UTAMA', yB_te, pred_llm_B)]:
    print(f'=== LLM API (Gemini {GEMINI_MODEL}) — {tag} ===')
    print(classification_report(yt, yp, labels=LABELS, zero_division=0))
    print('Confusion Matrix [baris=aktual, kolom=prediksi; urutan negatif, positif]:\n', confusion_matrix(yt, yp, labels=LABELS))

### 6.3 Tabel Perbandingan Ringkasan

Selain Split A dan B, ditambahkan evaluasi **suplemen** agar angka lebih stabil (test Split B hanya 8 teks unik): model klasik dievaluasi dengan *leave-one-text-out* (LOO atas 40 teks unik — tiap teks diprediksi oleh model yang tidak pernah melihat teks itu), dan Gemini pada seluruh 40 teks unik.

In [ ]:
# Suplemen: leave-one-text-out atas 40 teks unik
Xu, yu = uniq.review_text.reset_index(drop=True), uniq.sentiment.reset_index(drop=True)
pred_loo = []
for tr, te in LeaveOneOut().split(Xu):
    m = make_clf().fit(Xu[tr], yu[tr]); pred_loo.append(m.predict(Xu[te])[0])
pred_llm_u = Xu.map(llm_map).values

rows = [
  {**evaluate('Klasik TF-IDF+LogReg', yA_te, pred_classic_A), 'Evaluasi': 'Split A (starter, n=%d, bocor)' % len(yA_te)},
  {**evaluate('Gemini zero-shot',     yA_te, pred_llm_A),     'Evaluasi': 'Split A (starter, n=%d, bocor)' % len(yA_te)},
  {**evaluate('Klasik TF-IDF+LogReg', yB_te, pred_classic_B), 'Evaluasi': 'Split B (UTAMA, n=%d baris / %d teks)' % (len(yB_te), yB_te.index.map(df.review_text).nunique())},
  {**evaluate('Gemini zero-shot',     yB_te, pred_llm_B),     'Evaluasi': 'Split B (UTAMA, n=%d baris / %d teks)' % (len(yB_te), yB_te.index.map(df.review_text).nunique())},
  {**evaluate('Klasik TF-IDF+LogReg', yu, pred_loo),          'Evaluasi': 'Suplemen LOO (n=40 teks unik)'},
  {**evaluate('Gemini zero-shot',     yu, pred_llm_u),        'Evaluasi': 'Suplemen LOO (n=40 teks unik)'},
]
summary = pd.DataFrame(rows)[['Evaluasi','Pendekatan','Accuracy','Precision (macro)','Recall (macro)','F1 (macro)']]
summary.iloc[:, 2:] = summary.iloc[:, 2:].astype(float).round(3)
summary.to_csv('../documentation/model_comparison_table.csv', index=False)
print(summary.to_string(index=False))

# Gambar ringkasan
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
sB = summary[summary.Evaluasi.str.startswith('Split B')].set_index('Pendekatan')[['Accuracy','Precision (macro)','Recall (macro)','F1 (macro)']]
sB.T.plot.bar(ax=axes[0], rot=20, ylim=(0, 1.05)); axes[0].set_title('Metrik — Split B (utama)')
for ax, (nm, yp) in zip(axes[1:], [('Klasik', pred_classic_B), ('Gemini', pred_llm_B)]):
    ConfusionMatrixDisplay(confusion_matrix(yB_te, yp, labels=LABELS), display_labels=LABELS).plot(ax=ax, colorbar=False)
    ax.set_title(f'Confusion Matrix {nm} — Split B')
plt.tight_layout(); plt.savefig('../documentation/model_comparison_summary.png', dpi=130)
print('Gambar tersimpan.')

In [ ]:
# Contoh kesalahan (Split B) untuk analisis kualitatif
err = pd.DataFrame({'review_text': XB_te.values, 'aktual': yB_te.values, 'klasik': pred_classic_B, 'gemini': pred_llm_B}).drop_duplicates('review_text')
err[(err.aktual != err.klasik) | (err.aktual != err.gemini)].to_string(index=False)

In [ ]:
# Biaya & kecepatan (terukur)
t0 = time.time(); _ = make_clf().fit(XB_tr, yB_tr).predict(XB_te); t_classic = time.time() - t0
print(f'Klasik: train+prediksi {len(XB_te)} baris = {t_classic*1000:.1f} ms total, tanpa biaya API')
lat = cache.drop_duplicates('batch_no').batch_latency_s
print(f'Gemini: {len(lat)} request untuk {len(cache)} teks; latensi rata-rata {lat.mean():.1f} s/request (~{lat.mean()/BATCH:.1f} s/teks)')

## 7. Analisis Trade-off dan Limitation

**Performa (hasil eksperimen nyata, lihat Section 6.3):**
- **Split A (sesuai starter):** kedua pendekatan 100% — tapi angka ini **tidak bisa dipercaya untuk model klasik**: 98% baris test punya teks identik di train, jadi model cuma menghafal. Itulah alasan Split B dibuat.
- **Split B (utama, tanpa kebocoran):** Gemini Accuracy/Precision/Recall/F1 = 1.00; klasik Accuracy 0.857, macro-F1 0.838. Semua kesalahan klasik berasal dari **satu teks unik** ("Pelayanan lambat dan tidak responsif saat dikomplain.", 5 baris) yang diprediksi positif padahal negatif — jadi di level teks klasik benar 7 dari 8.
- **Suplemen LOO (40 teks unik):** Gemini 1.00 vs klasik 0.70 — gap lebih jelas karena sampelnya lebih banyak dan tiap teks diuji tanpa pernah dilihat model.

**Kenapa klasik kalah (dugaan, tidak diuji lebih jauh):** TF-IDF hanya melihat kata yang muncul di data latih (~32 teks unik) dan tidak memahami negasi. Teks yang salah berisi *pelayanan* (muncul di 2 dari 3 teks unik positif yang memuatnya) dan *responsif* (terbagi 1 positif : 1 negatif), kata yang tidak membawa sinyal negatif kuat, sedangkan frasa *tidak responsif* tidak ditangkap sebagai satu makna. LLM memahami makna & negasi tanpa data latih.

**Trade-off:**
| Aspek | Klasik (TF-IDF+LogReg) | Gemini (zero-shot) |
|---|---|---|
| Effort awal | Perlu data berlabel + training | Cukup menulis prompt |
| Kecepatan | ~8 ms untuk train+prediksi 35 baris | ~8.7 s per request batch 10 teks (~0.9 s/teks) |
| Biaya | Gratis, jalan lokal | Berbayar per token; free tier dibatasi **20 request/hari/model** (terkena sendiri saat eksperimen) |
| Skala data | Butuh data latih lebih banyak agar bagus | Tidak butuh data latih |
| Reproducibility | Deterministik penuh | Model bisa dipensiunkan (2.0-flash & 2.5-flash sudah 404), output bisa berubah antar versi |
| Privasi | Data tetap lokal | Teks ulasan dikirim ke pihak ketiga |

**Keterbatasan eksperimen (jujur):**
1. Dataset hanya **40 teks unik** (160 dari 200 baris duplikat) — angka sempurna Gemini pada 40 teks sederhana belum membuktikan performa di ulasan nyata yang ambigu, sarkas, atau campur bahasa/slang.
2. Test Split B hanya 8 teks unik; satu teks salah mengubah akurasi ~14 poin. Perbedaan kecil sangat rentan noise, tidak ada interval kepercayaan.
3. Semua label sangat jelas (positif/negatif tegas, tanpa netral) — LLM diuntungkan; kasus sulit tidak teruji.
4. Gemini dites dengan **batch prompting** (10 ulasan/request) karena kuota — bukan prompt per-ulasan seperti starter. Ada kemungkinan sedikit perbedaan hasil dibanding per-ulasan.
5. Satu model, satu prompt, satu run (temperature 0); tidak diuji variasi prompt atau few-shot. Contoh pembanding klasik lain (misal n-gram, SVM, atau model IndoBERT) tidak dicoba.
6. Model Gemini yang dipakai (`gemini-3.7-flash`) berbeda dari yang disebut di brief karena model itu sudah tidak tersedia.

## 8. Rekomendasi Technical Approach

**Rekomendasi: gunakan LLM (Gemini) sebagai pendekatan awal / prototipe, dengan jalur migrasi ke model klasik terlatih bila volume dan data berlabel sudah cukup.**

**Alasan (berdasarkan hasil di atas):**
1. Pada data tanpa kebocoran, Gemini zero-shot mengungguli klasik (1.00 vs 0.857 pada Split B; 1.00 vs 0.70 pada LOO) **tanpa** data latih — cocok untuk kondisi saat ini: data berlabel sangat sedikit (40 teks unik).
2. Model klasik di dataset ini praktis hanya baik karena menghafal (100% di Split A bocor); untuk teks baru performanya turun. Mengandalkannya sekarang berisiko.
3. Namun klasik jauh lebih murah dan cepat (~ms, lokal, tanpa kuota), sehingga untuk volume besar atau data sensitif lebih layak **setelah** dikumpulkan data berlabel jauh lebih banyak dan beragam (idealnya ratusan–ribuan ulasan unik).

**Langkah lanjutan yang disarankan:** (a) kumpulkan ulasan nyata yang beragam termasuk kasus ambigu/netral/sarkas dan evaluasi ulang; (b) uji prompt per-ulasan dan few-shot dengan kuota berbayar; (c) pakai Gemini untuk membantu pelabelan lalu latih model klasik/IndoBERT dan bandingkan lagi; (d) pantau biaya & versi model.

**Catatan kehati-hatian:** rekomendasi ini berlaku untuk dataset kecil & mudah ini; belum cukup bukti untuk menyimpulkan LLM selalu lebih baik.